In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import pickle
from pathlib import Path

import anndata as ad
import numpy as np
import scipy
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scib_metrics

sc.settings.set_figure_params(dpi=200, frameon=False)
sc.set_figure_params(dpi=500)
plt.rcParams['figure.figsize'] = (5, 5)

In [3]:
from metrics_lib import *

In [4]:
DATASET_KEY = 'dataset'
SAMPLE_KEY = 'sample'
EMB_KEY = 'X_scanvi_emb'
REF_SAMPLE_FRAC = 1.
# REF_SAMPLE_FRAC = 0.1

# REF_DATASET_SUBSET = ['Meyer_2019', 'Teichmann_Meyer_2019']
REF_DATASET_SUBSET = []
# QUERY_DATASET_SUBSET = ['Meyer_2021_3prime', 'Regev_2021_Nuclei']
QUERY_DATASET_SUBSET = []

HLCA_DATA_PATH = os.path.expanduser('~/data/HLCA/hlca_all_cellxgene.h5ad')
METRIC_OUT_PATH = os.path.expanduser('~/io/q2r_metric/')
OUTPUT_FORMAT = f"metric_results_ref_frac_{REF_SAMPLE_FRAC}" + "_query_{query_dataset}_metric_{metric}.pkl"

In [5]:
adata = sc.read(HLCA_DATA_PATH, backed='r')
adata

AnnData object with n_obs × n_vars = 2282447 × 56295 backed at '/dss/dsshome1/03/di93zoz/data/HLCA/hlca_all_cellxgene.h5ad'
    obs: 'suspension_type', 'donor_id', 'is_primary_data', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'tissue_ontology_term_id', 'organism_ontology_term_id', 'sex_ontology_term_id', "3'_or_5'", 'BMI', 'age_or_mean_of_age_range', 'age_range', 'anatomical_region_ccf_score', 'ann_coarse_for_GWAS_and_modeling', 'ann_finest_level', 'ann_level_1', 'ann_level_2', 'ann_level_3', 'ann_level_4', 'ann_level_5', 'cause_of_death', 'core_or_extension', 'dataset', 'fresh_or_frozen', 'log10_total_counts', 'lung_condition', 'mixed_ancestry', 'original_ann_level_1', 'original_ann_level_2', 'original_ann_level_3', 'original_ann_level_4', 'original_ann_level_5', 'original_ann_nonharmonized', 'reannotation_type', 'sample', 'scanvi_label', 'sequencing_platform', 's

In [6]:
reference_obs = adata.obs[adata.obs['core_or_extension'] == 'core']
if len(REF_DATASET_SUBSET) > 0:
    reference_obs = reference_obs[reference_obs[DATASET_KEY].isin(REF_DATASET_SUBSET)]

reference_obs[DATASET_KEY].cat.remove_unused_categories().value_counts()

dataset
Meyer_2019              35554
Teichmann_Meyer_2019    12231
Name: count, dtype: int64

In [7]:
query_all_obs = adata.obs[adata.obs['core_or_extension'] == 'extension']
if len(QUERY_DATASET_SUBSET) > 0:
    query_all_obs = query_all_obs[query_all_obs[DATASET_KEY].isin(QUERY_DATASET_SUBSET)]
query_all_obs[DATASET_KEY].cat.remove_unused_categories().value_counts()

dataset
Regev_2021_Nuclei    89426
Meyer_2021_3prime    24628
Name: count, dtype: int64

In [8]:
query_keys = query_all_obs[DATASET_KEY].cat.remove_unused_categories().value_counts().index[::-1]
query_keys

CategoricalIndex(['Meyer_2021_3prime', 'Regev_2021_Nuclei'], categories=['Meyer_2021_3prime', 'Regev_2021_Nuclei'], ordered=False, dtype='category', name='dataset')

In [9]:
def is_already_calculated(qkey, output_schema):
    already_calculated = True
    for key in output_schema:
        filename = os.path.join(METRIC_OUT_PATH, OUTPUT_FORMAT.format(query_dataset=qkey, metric=key))
        if os.path.exists(filename):
            pass
            # print(f"File already exists: {filename}")
        else:
            already_calculated = False
    return already_calculated

def save_output(qkey, output):
    for key in output:
        filename = os.path.join(METRIC_OUT_PATH, OUTPUT_FORMAT.format(query_dataset=qkey, metric=key))
        if os.path.exists(filename):
            print(f"WARNING: File already exists: {filename}")
        with open(filename, 'wb') as f:
            pickle.dump(output[key], f)

if REF_SAMPLE_FRAC < 1.:
    np.random.seed(0)
    current_reference_obs = reference_obs.sample(frac=REF_SAMPLE_FRAC)
else:
    current_reference_obs = reference_obs

In [10]:
tasks = {
    'scib': lambda adata_all, adata_query, dry_run: calculate_scib_metrics(
        adata_all, EMB_KEY, 'core_or_extension', 
        dataset_key=DATASET_KEY, sample_key=SAMPLE_KEY, dry_run=dry_run),
    # Note: adata_query.X should be log normalized.
    'symphony': lambda adata_all, adata_query, dry_run: calculate_symphony_metrics(
        adata_all, adata_query, EMB_KEY, 'donor_id',
        dataset_key=DATASET_KEY, sample_key=SAMPLE_KEY, dry_run=dry_run),
    'other': lambda adata_all, adata_query, dry_run: calculate_other_metrics(
        adata_all, adata_query, EMB_KEY,
        dataset_key=DATASET_KEY, sample_key=SAMPLE_KEY, dry_run=dry_run),
}
taks_keys = list(tasks.keys())

In [11]:
def run(qkey, task, dry_run=False):
    if not dry_run:
        current_query_obs = query_all_obs[query_all_obs[DATASET_KEY] == qkey].copy()
        metric_obs = pd.concat([current_reference_obs, current_query_obs])
        adata_all = adata[metric_obs.index]
        adata_query = adata[metric_obs[metric_obs['core_or_extension'] == 'extension'].index]
    else:
        adata_all, adata_query = None, None

    output_schema = task(adata_all, adata_query, True)
    already_calculated = is_already_calculated(qkey, output_schema)
    if dry_run:
        return already_calculated
    if not already_calculated:
        try:
            output = task(adata_all, adata_query, False)
            save_output(qkey, output)
        except Exception as e:
            print(e)
            return False
    return True

In [12]:
remaining_runs = np.ones((len(query_keys), len(taks_keys)))
for i, qkey in enumerate(query_keys):
    for j, tkey in enumerate(taks_keys):
        task = tasks[tkey]
        if run(qkey, task, dry_run=True):
            remaining_runs[i, j] = 0

print(remaining_runs.shape)
print(f"{remaining_runs.sum()} out of {np.ones_like(remaining_runs).sum()} remaining.")

# Random task picking strategy
while remaining_runs.sum() > 0:
    print(f"{remaining_runs.sum()} out of {np.ones_like(remaining_runs).sum()} remaining.")
    i = np.random.choice(range(len(query_keys)))
    j = np.random.choice(range(len(taks_keys)))
    qkey = query_keys[i]
    tkey = taks_keys[j]
    task = tasks[tkey]
    if run(qkey, task, dry_run=True):
        remaining_runs[i, j] = 0
        print(f"{tkey} for {qkey} already calculated. Skipping.")
        continue
    else:
        print(f"Running {tkey} for {qkey}.")
        if run(qkey, task, dry_run=False):
            remaining_runs[i, j] = 0

(36, 3)
0.0 out of 108.0 remaining.


In [42]:
query_items = {
    'per_dataset': query_all_obs['dataset'].cat.remove_unused_categories().unique(),
    'per_sample': query_all_obs['sample'].cat.remove_unused_categories().unique(),
    'per_cell': query_all_obs.index,
}
query_items

{'per_dataset': ['Gomperts_2021_CSMC', 'Sun_2020_batch3', 'Schiller_2020', 'Thienpont_2018_10Xv2', 'Regev_2021_Fresh', ..., 'Sun_2020_batch1', 'Sun_2020_batch2', 'Gomperts_2021_CFF', 'Sun_2020_batch4', 'Regev_2021_Cryo']
 Length: 36
 Categories (36, object): ['Banovich_Kropski_2020', 'Budinger_2020', 'Duong_lungMAP_unpubl', 'Eils_2020', ..., 'Wunderink_2021_cryo', 'Wunderink_2021_fresh', 'Xu_2020_LAM1_3', 'Zhang_2021'],
 'per_sample': ['cc05p', 'GSM4906348', 'muc9826', 'scrBT1428', 'D12_4', ..., 'WTDAtest7888001', 'BT1245', 'BG9_35', 'Pool_440', 'Pool_441']
 Length: 565
 Categories (565, object): ['001C', '002C', '003C', '010I', ..., 'scrBT1429', 'scrBT1430', 'scrBT1431', 'scrBT1432'],
 'per_cell': Index(['cc05p_CATGCCTGTGTGCCTG_carraro_csmc',
        'D062_TGACCCTTCAAACCCA-sub_wang_sub_batch3',
        'muc9826_GTCGTGAGAGGA_mayr', 'scrBT1428_CCACGGATCGCATGGC_lambrechts',
        '04-P103142-S149-R01-GCAGCCAGTACGTAGG_delorey_fresh',
        'GCGAGAAGTCACTTCC-CV001_KM8853547-ucl_meyer_n

In [47]:
dataframes_to_save = {}
query_items = {
    'per_dataset': query_all_obs['dataset'].cat.remove_unused_categories().unique(),
    'per_sample': query_all_obs['sample'].cat.remove_unused_categories().unique(),
    'per_cell': query_all_obs.index,
}


for i, qkey in enumerate(query_keys):
    for j, tkey in enumerate(taks_keys):
        task = tasks[tkey]
        output_schema = task(None, None, True)
        for key in output_schema:
            filename = os.path.join(METRIC_OUT_PATH, OUTPUT_FORMAT.format(query_dataset=qkey, metric=key))
            if not os.path.exists(filename):
                print(f"Error: File Not Found!: {filename}")
                continue
            with open(filename, 'rb') as f:
                print(f"Reading {filename}")
                output = pickle.load(f)
                # print(output)
            for grouping in output:
                df = output[grouping]
                if isinstance(df, pd.Series):
                    df = pd.DataFrame({'value': df, 'metric': key, 'grouping': grouping}, index=df.index)
                if not isinstance(df, pd.DataFrame):
                    print(f"{grouping} is of type {type(df)} not pd.DataFrame.")
                    continue
                
                df = df[df.index.isin(query_items[grouping])].copy()
                if grouping == 'per_dataset':
                    df = df[df.index.isin([qkey])].copy()
                save_key = f"{key}__{grouping}"
                if save_key not in dataframes_to_save:
                    dataframes_to_save[save_key] = df
                else:
                    dataframes_to_save[save_key] = pd.concat([dataframes_to_save[save_key], df])

Reading /dss/dsshome1/03/di93zoz/io/q2r_metric/metric_results_ref_frac_1.0_query_Regev_2021_Fresh_metric_asw_batch.pkl
Reading /dss/dsshome1/03/di93zoz/io/q2r_metric/metric_results_ref_frac_1.0_query_Regev_2021_Fresh_metric_lisi_batch.pkl
Reading /dss/dsshome1/03/di93zoz/io/q2r_metric/metric_results_ref_frac_1.0_query_Regev_2021_Fresh_metric_kbet_batch.pkl
Reading /dss/dsshome1/03/di93zoz/io/q2r_metric/metric_results_ref_frac_1.0_query_Regev_2021_Fresh_metric_lisi_query_donor.pkl
Reading /dss/dsshome1/03/di93zoz/io/q2r_metric/metric_results_ref_frac_1.0_query_Regev_2021_Fresh_metric_wiq_knn_corr.pkl
Reading /dss/dsshome1/03/di93zoz/io/q2r_metric/metric_results_ref_frac_1.0_query_Regev_2021_Fresh_metric_cluster_preservation_score.pkl
total_score is of type <class 'float'> not pd.DataFrame.
Reading /dss/dsshome1/03/di93zoz/io/q2r_metric/metric_results_ref_frac_1.0_query_Thienpont_2018_10Xv1_metric_asw_batch.pkl
Reading /dss/dsshome1/03/di93zoz/io/q2r_metric/metric_results_ref_frac_1.0_qu

In [48]:
dataframes_to_save

{'asw_batch__per_cell':                                                        value     metric  \
 04-P103142-S149-R01-GCAGCCAGTACGTAGG_delorey_fresh  0.623470  asw_batch   
 04-P103142-S149-R01-TGATTTCAGTTCATGC_delorey_fresh  0.648899  asw_batch   
 04-P103142-S149-R01-CTGCCTATCTCCGCAT_delorey_fresh  0.678393  asw_batch   
 04-P103142-S150-R01-AAAGGATAGGTGAGCT_delorey_fresh  0.799982  asw_batch   
 04-P103142-S149-R01-TTACTGTTCTCGGCTT_delorey_fresh  0.693138  asw_batch   
 ...                                                      ...        ...   
 022C-b_TCGCGAGAGGACACCA_adams                       0.964096  asw_batch   
 152CO-a_CCACTACCACACTGCG_adams                      0.867506  asw_batch   
 222C_TCGTACCAGACAGACC_adams                         0.942139  asw_batch   
 022C-b_GGATGTTTCCAAGTAC_adams                       0.849638  asw_batch   
 145I-a_GTCGTAACAGTAGAGC_adams                       0.903845  asw_batch   
 
                                                     grouping  

In [49]:
for save_key in dataframes_to_save:
    dataframes_to_save[save_key].to_csv(f"./results/{save_key}.csv", index=True)

In [ ]:
# # Full running strategy
# for qkey in query_keys:
#     print(qkey)
#     current_query_obs = query_all_obs[query_all_obs[DATASET_KEY] == qkey].copy()
#     metric_obs = pd.concat([current_reference_obs, current_query_obs])
#     adata_all = adata[metric_obs.index]
#     adata_query = adata[metric_obs[metric_obs['core_or_extension'] == 'extension'].index]

#     for x in :
#         output_schema = x(True)
#         already_calculated = is_already_calculated(qkey, output_schema)
#         if not already_calculated:
#             output = x(False)
#             save_output(qkey, output_schema)

Regev_2021_Fresh
Time to complete scib neighbors calculation: 0.000 s
Time to complete asw_batch: 0.000 s
Time to complete lisi_batch: 0.000 s
Time to complete kbet_batch: 0.000 s
File already exists: /dss/dsshome1/03/di93zoz/io/q2r_metric/metric_results_ref_frac_1.0_query_Regev_2021_Fresh_metric_asw_batch.pkl
File already exists: /dss/dsshome1/03/di93zoz/io/q2r_metric/metric_results_ref_frac_1.0_query_Regev_2021_Fresh_metric_lisi_batch.pkl
File already exists: /dss/dsshome1/03/di93zoz/io/q2r_metric/metric_results_ref_frac_1.0_query_Regev_2021_Fresh_metric_kbet_batch.pkl
Time to complete symphony metrics neighbors calculation: 0.000 s
Time to complete lisi_query_donor: 0.000 s
Time to complete wiq_knn_corr: 0.000 s
File already exists: /dss/dsshome1/03/di93zoz/io/q2r_metric/metric_results_ref_frac_1.0_query_Regev_2021_Fresh_metric_lisi_query_donor.pkl
File already exists: /dss/dsshome1/03/di93zoz/io/q2r_metric/metric_results_ref_frac_1.0_query_Regev_2021_Fresh_metric_wiq_knn_corr.pkl
T

No GPU/TPU found, falling back to CPU. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


Time to complete scib neighbors calculation: 535.314 s
